In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shrutiy07/spectrogram/gw_dataset_final_processed.h5
/kaggle/input/datasets/shrutiy07/spectrogram/gw_spectrogram_dataset.h5


In [2]:
import os 
import random
import numpy as np 
import h5py
import torch 
import torch.nn as nn
from torch. utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.signal import spectrogram
from sklearn.metrics import(roc_auc_score,f1_score,precision_score,accuracy_score,recall_score,confusion_matrix,)
from torchvision import models 

In [3]:
NORMALIZE = True
SPEC_KEY="spectrogram"
LABEL_KEY="label"
NORMALISE=True
N_EPOCHS=80
BATCH_SIZE=64
LEARNING_RATE=1e-4
WEIGHT_DECAY=1e-5
PATIENCE_LIMIT=15
MIN_EPOCHS=20
SEED=42
CHECKPOINT_PATH="best_resnet18_gw_model"

In [4]:
seed = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [5]:
def inspect_h5(h5_path):
    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"HDF5 file not found:\n{h5_path}")
    print("\nHDF5 dataset structure")
    with h5py.File(h5_path,"r") as f:
         def show (name ,obj):
             if isinstance(obj,h5py.Dataset):
                 print(f"{name}: shape={obj.shape},dtype={obj.dtype}")
         f.visititems(show)
    

In [6]:
def load_split(h5_path,split):
    with h5py.File(h5_path,"r")as f:
        if split not in f:
           raise KeyError(f"Group '{split}' not found.\n"f" Available groups: {list(f.keys())}")
        group=f[split]
        if SPEC_KEY not in group:
            raise KeyError(f"Dataset '{SPEC_KEY}' not found in '{split}'.\n"
                f"Available datasets: {list(group.keys())}")

        if LABEL_KEY not in group:
            raise KeyError(
                f"Dataset '{LABEL_KEY}' not found in '{split}'.\n"
            f"Available datasets: {list(group.keys())}")
        specs=group[SPEC_KEY][:]
        labels=group[LABEL_KEY][:]
        #Spectrogram dimension validation and single-channel normalization before ResNet preprocessing.
    if specs.ndim==4:
        if specs.shape[1]!=1:
            raise ValueError("Expected a single-channel spectrogram when "
                f"using 4D input, but got shape {specs.shape}")
        specs=specs[:,0]  #remove unnecessary channel
    if specs.ndim != 3:
        raise ValueError(
            f"Expected spectrograms shaped (N, H, W), "
            f"but received {specs.shape}")

    specs = specs.astype(np.float32)
    labels = labels.astype(np.int64)

    unique_label=np.unique(labels)  #all different values
    if not np.all(np.isin(unique_label, [0, 1])):
        raise ValueError(f"Labels must contain only 0 and 1.\n "f"Found labels: {unique_labels}")
#check for nan/inf
    if not np.isfinite(specs).all():
        raise ValueError(f"{split} spectrograms contain NaN or Inf values.")
#check dataset dimensions
    if specs.shape[1] < 32 or specs.shape[2] < 32:
        raise ValueError( f"Spectrogram spatial size is too small:"
            f"{specs.shape[1:]}." f"Use at least approximately 32x32 for ResNet-18.")
    return specs ,labels

In [7]:
def fit_normalization(train_specs):
    """
    Calculate normalization statistics using TRAIN ONLY.
    """
    mean = float(train_specs.mean())
    std = float(train_specs.std())
    # Prevent division by zero.
    if std < 1e-8:
        raise ValueError("Training spectrogram standard deviation is almost zero.")
    return {"mean": mean,"std": std,}

def apply_normalization(specs, params):
    """
    Apply training-set normalization to any split.
    """
    return (specs - params["mean"]) / params["std"]


In [8]:
class GWSpectrogramDataset(Dataset):

    def __init__(self, specs, labels): # (N, H, W)-> (N, 1, H, W)
        self.specs = torch.from_numpy(specs).unsqueeze(1)
        self.labels = torch.from_numpy(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        x = self.specs[index]
        y = self.labels[index]
        return x, y

def build_pretrained_resnet18(dropout=0.2):
    """The original ResNet expects:
        (N, 3, H, W)
    Our spectrograms are:
        (N, 1, H, W)
    Therefore, the single spectrogram channel is repeated to three channels during the forward pass.
    This keeps the original pretrained ResNet convolutional layers unchanged.
    The final ImageNet classifier is replaced by:
        Dropout -> Linear -> 2 classes"""

    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    # Save original forward
    original_forward = model.forward
    def forward_with_grayscale(x):     # x: (B, 1, H, W)
        if x.ndim != 4:
            raise ValueError(f"Expected input shape (B, C, H, W), got {x.shape}")

        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        elif x.shape[1] != 3:
            raise ValueError(f"Expected 1 or 3 input channels, got {x.shape[1]}" )

        return original_forward(x)

    model.forward = forward_with_grayscale
     # Replace ImageNet classification head
    model.fc = nn.Sequential(
        nn.Dropout(dropout),nn.Linear(model.fc.in_features,2))
    return model

In [9]:
def train_one_epoch(model,loader,optimizer,criterion, device,grad_clip=1.0,):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(),grad_clip )
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == y).sum().item()
        total += x.size(0)

    return (total_loss / total,correct / total)

In [10]:
def evaluate_with_metrics(model,loader,criterion,device,):
    model.eval()
    all_probs=[]
    all_preds=[]
    all_labels=[]
    total_loss=0.0
    total=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)
            outputs=model(x)
            loss=criterion(outputs,y)
            total_loss += loss.item()*x.size(0)
            total+= x.size(0)
            probs = torch.softmax(outputs,dim=1)[:, 1]
            preds = outputs.argmax(dim=1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)   
#AUC safety check
    if len(np.unique(all_labels)) == 2:
        auc = roc_auc_score(all_labels,all_probs)
    else:
        auc = float("nan")
    metrics = {"loss": total_loss / total,
       "accuracy": accuracy_score(all_labels,all_preds),"auc": auc,"f1": f1_score(all_labels,all_preds,
            zero_division=0), "precision": precision_score(all_labels,all_preds,zero_division=0 ),
    "recall": recall_score(all_labels,all_preds,zero_division=0),
    "confusion_matrix": confusion_matrix(all_labels, all_preds),}
    return metrics

In [11]:
def run_training( h5_path, n_epochs=80,batch_size=64,lr=1e-4,weight_decay=1e-5,patience_limit=3,
    min_epochs=5,monitor="auc",checkpoint_path="best_resnet_model.pt",):

    set_seed(SEED)
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_specs, y_train = load_split( h5_path,"train")

    val_specs, y_val = load_split(h5_path,"val")

    test_specs, y_test = load_split(h5_path, "test")

    print("Spectrogram shape:",train_specs.shape[1:])

    print(f"Examples -> " f"train: {len(y_train)}, "
        f"val: {len(y_val)}, "
        f"test: {len(y_test)}")

    norm_params = None
    if NORMALIZE:
        # Calculate mean/std using TRAIN ONLY
        norm_params = fit_normalization(train_specs)
        print("Normalization parameters " "(TRAIN ONLY):",norm_params)
        # Apply SAME train statistics to all splits
        train_specs = apply_normalization(train_specs,norm_params)
        val_specs = apply_normalization( val_specs,norm_params )
        test_specs = apply_normalization(test_specs, norm_params)

    
    #Class distribution  
    class_counts = np.bincount( y_train, minlength=2 )
    if np.any(class_counts == 0):
        raise ValueError("Both classes must be present in "
            "the training set.\n"
            f"Train class counts: {class_counts}")
    print("Training class counts:\n"
        f"  Noise (0): {class_counts[0]}\n"
        f"  GW    (1): {class_counts[1]}" )

    #  Class-weighted loss
    class_weights_np = (class_counts.sum()/(2.0 * class_counts) )
    class_weights = torch.tensor(class_weights_np,dtype=torch.float32,device=device)
    print("Class weights:", class_weights_np )

    #  Datasets
    train_ds = GWSpectrogramDataset(train_specs,y_train)
    val_ds = GWSpectrogramDataset(val_specs,y_val)
    test_ds = GWSpectrogramDataset(test_specs,y_test)

    # DataLoaders 
    pin_memory = (device.type == "cuda")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,num_workers=0, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds,batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size,  shuffle=False,  num_workers=0, pin_memory=pin_memory )

    # Pretrained ResNet-18
    print( "\nModel: ImageNet-pretrained ResNet-18")
    model = build_pretrained_resnet18(dropout=0.2).to(device)

    # Loss
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    #  Optimizer
    optimizer = torch.optim.Adam( model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="max",factor=0.5,patience=6 )

    if monitor not in ["auc", "accuracy"]:
        raise ValueError("monitor must be either 'auc' or 'accuracy'" )
    print(f"Early stopping metric: "f"{monitor}")

    best_metric = -float("inf")
    best_epoch = 0
    patience_counter = 0

    # Training loop   
    for epoch in range(1,n_epochs + 1):
        # Training
        train_loss, train_acc = train_one_epoch( model,train_loader,optimizer,criterion, device)
        # Validation
        val_m = evaluate_with_metrics(model,val_loader,criterion,device )

        if monitor == "auc":
            current_metric = val_m["auc"]
            metric_name = "val_auc"
        else:
            current_metric = val_m["accuracy"]
            metric_name = "val_accuracy"

        if not np.isnan(val_m["auc"]):
            scheduler.step(val_m["auc"] )

        # Check improvement
        improved = ( not np.isnan(current_metric)
            and current_metric > best_metric)
        if improved:
            best_metric = float( current_metric)
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(),checkpoint_path)

        else:
            patience_counter += 1

        # Current learning rate
        current_lr = ( optimizer.param_groups[0]["lr"] )
        print( f"Epoch {epoch:3d}/{n_epochs} | "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.3f} | "
            f"val_loss={val_m['loss']:.4f} "
            f"val_acc={val_m['accuracy']:.3f} "
            f"val_auc={val_m['auc']:.3f} "
            f"val_f1={val_m['f1']:.3f} "
            f"val_prec={val_m['precision']:.3f} "
            f"val_rec={val_m['recall']:.3f} "
            f"| {metric_name}="
            f"{current_metric:.3f} "
            f"| lr={current_lr:.2e}")

        # Early stopping
        if (epoch >= min_epochs
            and patience_counter >= patience_limit):
            print(f"\nEarly stopping at epoch " f"{epoch}." )
            print( f"No improvement in "
                f"{monitor} for "
                f"{patience_limit} epochs." )
            break

    # Load BEST checkpoint
    if not os.path.exists( checkpoint_path):
        raise FileNotFoundError(
            f"Best model was not saved at:\n"
            f"{checkpoint_path}")
    best_state = torch.load( checkpoint_path, map_location=device, weights_only=True )
    model.load_state_dict(
        best_state )

    #  Final test evaluation
    test_metrics = evaluate_with_metrics( model, test_loader,criterion,device)
    print( f"Best {monitor}: "  f"{best_metric:.4f}")
    print( f"Best epoch: " f"{best_epoch}" )
    print("\nTEST RESULTS")
    print(  f"Loss      : " f"{test_metrics['loss']:.4f}")
    print(f"Accuracy  : "f"{test_metrics['accuracy']:.4f}" )
    print( f"AUC       : " f"{test_metrics['auc']:.4f}")
    print( f"F1        : "f"{test_metrics['f1']:.4f}")
    print(f"Precision : " f"{test_metrics['precision']:.4f}")
    print( f"Recall    : " f"{test_metrics['recall']:.4f}")
    print("\nConfusion Matrix")
    print("(rows = true class, ""columns = predicted class)" )
    print(test_metrics["confusion_matrix"])
    return (model,test_metrics,best_epoch,best_metric)

In [12]:
H5_PATH = ("/kaggle/input/datasets/shrutiy07/spectrogram/gw_spectrogram_dataset.h5")
inspect_h5(H5_PATH)
model_auc, test_auc_metrics, auc_best_epoch, auc_best_metric = run_training(
    H5_PATH,
    n_epochs=80,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-5,
    patience_limit=3,
    min_epochs=5,
    monitor="auc",
    checkpoint_path="best_resnet_auc.pt")


HDF5 dataset structure
test/label: shape=(840,),dtype=int64
test/spectrogram: shape=(840, 129, 125),dtype=float32
train/label: shape=(3920,),dtype=int64
train/spectrogram: shape=(3920, 129, 125),dtype=float32
val/label: shape=(840,),dtype=int64
val/spectrogram: shape=(840, 129, 125),dtype=float32
Using device: cuda
Spectrogram shape: (129, 125)
Examples -> train: 3920, val: 840, test: 840
Normalization parameters (TRAIN ONLY): {'mean': -8.520856857299805, 'std': 3.3550610542297363}
Training class counts:
  Noise (0): 1120
  GW    (1): 2800
Class weights: [1.75 0.7 ]

Model: ImageNet-pretrained ResNet-18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


Early stopping metric: auc
Epoch   1/80 | train_loss=0.4723 train_acc=0.742 | val_loss=1.2910 val_acc=0.714 val_auc=0.882 val_f1=0.833 val_prec=0.714 val_rec=1.000 | val_auc=0.882 | lr=1.00e-04
Epoch   2/80 | train_loss=0.1000 train_acc=0.962 | val_loss=0.3216 val_acc=0.895 val_auc=0.990 val_f1=0.932 val_prec=0.873 val_rec=0.998 | val_auc=0.990 | lr=1.00e-04
Epoch   3/80 | train_loss=0.0483 train_acc=0.984 | val_loss=0.3070 val_acc=0.893 val_auc=0.996 val_f1=0.930 val_prec=0.871 val_rec=0.998 | val_auc=0.996 | lr=1.00e-04
Epoch   4/80 | train_loss=0.0292 train_acc=0.991 | val_loss=1.1168 val_acc=0.785 val_auc=0.988 val_f1=0.869 val_prec=0.769 val_rec=0.998 | val_auc=0.988 | lr=1.00e-04
Epoch   5/80 | train_loss=0.0166 train_acc=0.995 | val_loss=0.0945 val_acc=0.979 val_auc=0.996 val_f1=0.985 val_prec=0.979 val_rec=0.992 | val_auc=0.996 | lr=1.00e-04
Epoch   6/80 | train_loss=0.0229 train_acc=0.993 | val_loss=1.6651 val_acc=0.750 val_auc=0.989 val_f1=0.851 val_prec=0.741 val_rec=1.000 |

In [13]:
model_acc, test_acc_metrics, acc_best_epoch, acc_best_metric = run_training(
    H5_PATH,
    n_epochs=80,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-5,
    patience_limit=3,
    min_epochs=5,
    monitor="accuracy",
    checkpoint_path="best_resnet_accuracy.pt")

Using device: cuda
Spectrogram shape: (129, 125)
Examples -> train: 3920, val: 840, test: 840
Normalization parameters (TRAIN ONLY): {'mean': -8.520856857299805, 'std': 3.3550610542297363}
Training class counts:
  Noise (0): 1120
  GW    (1): 2800
Class weights: [1.75 0.7 ]

Model: ImageNet-pretrained ResNet-18
Early stopping metric: accuracy
Epoch   1/80 | train_loss=0.4723 train_acc=0.742 | val_loss=1.2910 val_acc=0.714 val_auc=0.882 val_f1=0.833 val_prec=0.714 val_rec=1.000 | val_accuracy=0.714 | lr=1.00e-04
Epoch   2/80 | train_loss=0.1000 train_acc=0.962 | val_loss=0.3216 val_acc=0.895 val_auc=0.990 val_f1=0.932 val_prec=0.873 val_rec=0.998 | val_accuracy=0.895 | lr=1.00e-04
Epoch   3/80 | train_loss=0.0483 train_acc=0.984 | val_loss=0.3070 val_acc=0.893 val_auc=0.996 val_f1=0.930 val_prec=0.871 val_rec=0.998 | val_accuracy=0.893 | lr=1.00e-04
Epoch   4/80 | train_loss=0.0292 train_acc=0.991 | val_loss=1.1168 val_acc=0.785 val_auc=0.988 val_f1=0.869 val_prec=0.769 val_rec=0.998 | 